# Machine Learning Model for Laptop Price Prediction
---

## Import Independencies or Libraries

In [ ]:
import pandas as pd  # Import pandas library for file handling
import numpy as np   # Import numpy library for numerical computations
import matplotlib.pyplot as plt  # Import matplotlib for visualization
import seaborn as sns # For visulization
import re  # Regular expression for feature engineering

import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")


## Dataset Overview

In [ ]:
#Load the laptop.csv file and create a dataframe
df= pd.read_csv('laptop_data.csv')

In [ ]:
#Display the dataset
df.head()

In [ ]:
# Display random values to find any abnormalities in the data

df.sample(10)

In [ ]:
# Shape of the dataset
df.shape

In [ ]:
# Display summary of the dataset structure
df.info()

---
## Data Abnormalities 

1. ScreenResolution Column: We can extract more features from the ScreenResolution column such as Touchscreen functionality, IPS availability, screen width and height, and then calculate Pixels per inch (PPI).
2. Converting RAM Column to Integer Data Type and removing the 'GB' suffix.
3. Memory Column Expansion: We can expand the Memory column to include or generate more features such as HDD, SSD, Flash Storage, and Hybrid storage.
4. Converting Weight Column to Float Data Type
5. Drop Unnamed Column from the dataset.
6. Extract processor from Cpu columns and categorize into different processor types.

Let's start by loading the data and addressing each of these tasks one by one.

---

## Data Cleansing

### Missing Count


In [ ]:
# Count of missing values
df.isnull().sum()

### Drop "Unnamed: 0"

In [ ]:
# Drop the column unnamaed: 0 because it is similar to index column
df.drop(columns=['Unnamed: 0'], inplace=True)

#Display the dataset after dropping the column
df.head()

## Feature Engineering
---

### Feature Engineering: RAM Column

RAM is a crucial feature when selecting a laptop, as prices typically increase with the capacity of RAM. However, the RAM column in our dataset has an object datatype because each value is suffixed with 'GB'. To utilize RAM as a numerical feature, we need to convert it to an integer datatype. This involves removing 'GB' from the end of each value and converting it to an integer datatype.

In [ ]:
# Remove GB from RAM
df['Ram']= df['Ram'].str.replace('GB','')

# Convert the 
df['Ram']= df['Ram'].astype(int)

# Check the data type of the RAM now.
df.info()

### Feature Engineering: Weight Column

Similarly, remove the 'kg' suffix from the Weight column and convert it to float.

In [ ]:
# Remove the Kg
df['Weight']=df['Weight'].str.replace('kg','')

# Convert it to float data type
df['Weight']=df['Weight'].astype(float)

# Check the datatype
df.info()

### Feature Engineering: ScreenResolution Column

The ScreenResolution column contains valuable information that can significantly impact the price prediction of a laptop. Important factors include whether the laptop has an IPS display, touchscreen functionality, and the resolution (width x height) of the screen.

To utilize this information effectively, we will extract each component from the ScreenResolution column and create new features for:
- IPS Display: Boolean feature indicating whether the laptop has an IPS display.
- Touchscreen: Boolean feature indicating whether the laptop has touchscreen functionality.
- Screen Resolution Width: Numerical feature representing the width of the screen resolution.
- Screen Resolution Height: Numerical feature representing the height of the screen resolution.

Note: In further steps screen resolution width and height will be used to calulate PPI (Pixel Per Inches) feature.

By extracting and creating these features, we aim to capture the hidden information within the ScreenResolution column, which can enhance the predictive power of our machine learning model. Later, we can drop the ScreenResolution Column from the dataframe.



In [ ]:
# Display the dataset to check the screenResolution column
df['ScreenResolution'].head(10)

In [ ]:
# Display the unique values count for each screen resolution values.
df['ScreenResolution'].value_counts()

### Extracting Touchscreen Feature

To extract the touchscreen feature from the ScreenResolution column:
- Assign 0 to represent non-touchscreen laptops.
- Assign 1 to represent laptops with touchscreen functionality.

By extracting this feature, we aim to create a binary indicator that captures whether each laptop has touchscreen functionality or not.

In [ ]:
# Create Touchscreen (TS) column using lambda function

# If 'Touchscreen' is present in the ScreenResolution, assign 1 (Yes), otherwise assign 0 (No)
df['TS'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0) # TS means Touchscreen

# Display a random sample of 5 rows from the DataFrame to verify the changes
df.sample(5)


In [ ]:
# Print count of touchscreen and non-touchscreen laptops
print('Count of the Touchscreen and Non-Touchscreen laptops:')
print(df['TS'].value_counts())

# Print total count
print('Total Count:', df['TS'].value_counts().sum())

### Extracting IPS Feature

In [ ]:
# Create Touchscreen (TS) columns using the lambda 
df['IPS']=df['ScreenResolution'].apply(lambda x:1 if 'IPS' in x else 0)

#Display the random sample
df.sample(5)

In [ ]:
df['IPS'].value_counts()

### Extracting Resolution and Generating PPI (Pixel Per Inches) Feature

To extract the resolution information from the ScreenResolution column and generate the PPI (Pixels Per Inch) feature:
1. Split the ScreenResolution values into width and height components.
2. Calculate the diagonal screen size using the width and height components.
3. Calculate the resolution in pixels by multiplying the width and height.
4. Calculate the PPI by dividing the resolution by the diagonal screen size.
5. Assign the calculated PPI values to a new column in the DataFrame.

By generating the PPI feature, we aim to quantify the pixel density of each laptop's screen, which can provide insights into the display quality and clarity.

#### PPI Calculations

In order to calculate the PPI, first find the diagonal resolution using below formula:

    diagonal_resolution = sqrt(width_pixels**2 + height_pixels**2)
    
Now, calculate PPI by dividing diagonal resolution by diagonal display size in inches

    ppi = diagonal_resolution / diagonal_inches
    

#### Example usage:
width_pixels = 2560
height_pixels = 1600
diagonal_inches = 13.3

ppi = calculate_ppi(width_pixels, height_pixels, diagonal_inches)
print("PPI:", ppi)

In [ ]:
df['Inches'].value_counts()

In [ ]:
# Conversion function
def convert_screen_size(size):
    if size in [10.1, 12.0, 11.6, 11.3]:
        return 11.6
    elif size in [12.3, 12.5, 13.3, 13.0, 13.5]:
        return 13.3
    elif size in [13.9, 14.0, 14.1]:
        return 14.0
    elif size in [15, 15.4, 15.6]:
        return 15.6
    else:
        return 17.3
    
    # Apply conversion
df['Inches'] = df['Inches'].apply(convert_screen_size)
df['Inches'].value_counts()

In [ ]:
# Separate dataframe to hold the extracted width and height from the screen resolutin column
# (r'(\d+)x(\d+) the expression identified the resolution and extract the resolution

Resolution=df['ScreenResolution'].str.extract(r'(\d+)x(\d+)', expand=True).astype(int)
Resolution.astype(int)

In [ ]:
# Extract width and height and add them as new columns to df
df['Width_pixels'] = Resolution[0]
df['Height_pixels'] = Resolution[1]

In [ ]:
# Calculation for ppi
df['PPI']= round((((df['Width_pixels']**2) + (df['Height_pixels']**2))**0.5)/df['Inches']).astype(float)

In [ ]:
# Dropm unneccary columns
df=df.drop(columns=['ScreenResolution', 'Width_pixels', 'Height_pixels'])

# Display the dataframe df
df.sample(10)

### Modifying TypeName Feature

In [ ]:
# Display the value count for TypeName column.
df['TypeName'].value_counts()

In [ ]:
# Replace values in the 'TypeName' column
df['TypeName'] = df['TypeName'].replace({'2 in 1 Convertible': 'Convertible', 'Netbook': 'Notebook'})

# Count the occurrences of each unique value after replacement
type_counts = df['TypeName'].value_counts()
print(type_counts)

### Extracting Processor Feature

In [ ]:
# Display the count of unique CPU values in the 'Cpu' column
print(df['Cpu'])

In [ ]:
# Extract the processor information from the 'Cpu' column and create a new 'Processor' column
df['Processor'] = df['Cpu'].apply(lambda x: " ".join(x.split()[0:3]))

# Display thecount of unique processor values in the 'Processor' column of the DataFrame 'df'
df['Processor'].value_counts()

In [ ]:
# Define a function to extract and categorize processor information
def Extract_processor(text):
    # Check if the processor is Intel Core i7, i5, or i3
    if text == 'Intel Core i7' or text == 'Intel Core i5' or text == 'Intel Core i3':
        return text
    else:
        # If the processor is an Intel processor but not i7, i5, or i3, categorize it as 'Other Intel Processor'
        if text.split()[0] == 'Intel':
            return 'Other Intel Processor'
        else:
            # If the processor is not Intel, categorize it as 'AMD Processor'
            return 'AMD Processor'

# Apply the Extract_processor function to the 'Processor' column and update the column with the processed values
df['Processor'] = df['Processor'].apply(Extract_processor)

# Display the count of each unique processor category after processing
df['Processor'].value_counts()

In [ ]:
# Drop the 'Cpu' column from the DataFrame
df.drop(columns=['Cpu'], inplace=True)

# Display the updated DataFrame
df

### Extracting HDD, SDD, Flash Storage(FS), and Hybrid (HBD)

In [ ]:
df['Memory'].value_counts()

In [ ]:
# Define regex patterns
patterns = {
    'SSD': r'(\d+(?:\.\d+)?(?:GB|TB))\s+SSD',
    'HDD': r'(\d+(?:\.\d+)?(?:TB|GB))\s+HDD',
    'FS': r'(\d+GB)\s+Flash\s+Storage',
    'HBD': r'(\d+(?:\.\d+)?(?:GB|TB))\s+Hybrid'
}

# Initialize columns
df['SSD'] = 0
df['HDD'] = 0
df['FS'] = 0
df['HBD'] = 0

# Update values based on regex patterns
for pattern_type, pattern in patterns.items():
    for idx, row in df.iterrows():
        match = re.search(pattern, row['Memory'])
        if match:
            df.at[idx, pattern_type] = match.group(1)

Let's break down the regular expressions used in the patterns:

1. `'(\d+(?:\.\d+)?(?:GB|TB))\s+SSD'`:

- `\d+`: Matches one or more digits (0-9).
- `(?:\.\d+)?`: `(?: ... )` is a non-capturing group used here. It matches a decimal point `\.` followed by one or more digits `\d+`. The `?` after the group makes it optional, allowing for decimal values like '1.0'.
- `(?:GB|TB)`: Another non-capturing group that matches either 'GB' or 'TB'.
- `\s+`: Matches one or more whitespace characters.
- `SSD`: Matches the literal string 'SSD'.

2. `'(\d+(?:\.\d+)?(?:TB|GB))\s+HDD'`:

- Similar to the first pattern, but it matches 'TB' or 'GB' before 'HDD'.

3. `'(\d+GB)\s+Flash\s+Storage'`:

- `\d+GB`: Matches one or more digits followed by 'GB'.
- `\s+`: Matches one or more whitespace characters.
- `Flash\s+Storage`: Matches the literal string 'Flash Storage', where `\s+` allows for whitespace between 'Flash' and 'Storage'.

4. `'(\d+(?:\.\d+)?(?:GB|TB))\s+Hybrid'`:

- Similar to the first two patterns, but it matches 'Hybrid' instead of 'SSD' or 'HDD'.

In summary, these patterns are designed to extract memory capacities (in either gigabytes or terabytes) from strings that describe different types of storage devices, such as SSD, HDD, Flash Storage, or Hybrid.

In [ ]:
# Display the DataFrame head
df.head()

In [ ]:
# Remove 'GB' and 'TB' suffix from the features and convert them to integer data type
df['SSD'] = df['SSD'].str.replace('GB', '').str.replace('TB', '')
df['HDD'] = df['HDD'].str.replace('GB', '').str.replace('TB', '')
df['FS'] = df['FS'].str.replace('GB', '').str.replace('TB', '')
df['HBD'] = df['HBD'].str.replace('GB', '').str.replace('TB', '')

# Convert the features to integer data type
df['SSD'] = df['SSD'].fillna(0).astype(int) 
df['HDD'] = df['HDD'].fillna(0).astype(float).round().astype(int)
df['FS'] = df['FS'].fillna(0).astype(int)
df['HBD'] = df['HBD'].fillna(0).astype(float).round().astype(int)

# Drop the 'Memory' column from the df and update the df
df = df.drop(columns='Memory')

# Display the updated df head
df.head()

### Extracting Operating System (OS)

In [ ]:
# Display the count of unique values in the 'OpSys' column of the DataFrame 'df'
df['OpSys'].value_counts()

In [ ]:
def Extract_OpSys(str):  
    # Function to categorize operating systems
    
    # Check if the operating system is Windows 10, Windows 10 S, or Windows 7
    if str == 'Windows 10' or str == 'Windows 10 S' or str == 'Windows 7':
        return 'WindowsOS'
    
    # Check if the operating system is Linux
    elif str == 'Linux':
        return 'LinuxOS'
    
    # Check if the operating system is macOS or Mac OS X
    elif str == 'macOS' or str == 'Mac OS X':
        return 'MacOS'
    
    # If the operating system is not recognized, categorize it as OtherOS
    else:
        return 'OtherOS'
    
    
# Apply the Extract_OpSys function to the 'OpSys' column and create a new column 'OS' with the extracted operating system categories
df['OS'] = df['OpSys'].apply(Extract_OpSys)

# Display the count of unique values in the 'OS' column of the DataFrame 'df'
df['OS'].value_counts()

In [ ]:
# Drop the OpSys because not needed in further analysis
df=df.drop(columns=['OpSys'])

# Display the update df
df

### Extracting Vedio Card(VC) Feature

In [ ]:
# Extract the first word from each entry in the 'Gpu' column and create a new column 'VC' with the extracted values
df['VC'] = df['Gpu'].apply(lambda x: x.split()[0])

# Drop the 'Gpu' column from the DataFrame and update the DataFrame
df = df.drop(columns='Gpu')

### Arranging Dataframe

In [ ]:
# Define the desired column order
desired_column_order = ['Company', 'OS','Processor','TypeName', 'VC', 'Ram',  'TS',  'Weight','FS', 'HBD', 'HDD','SSD', 'IPS', 'Inches', 'PPI','Price']

# Reorder the DataFrame with sorted column names
df = df[desired_column_order]

df.head()

In [ ]:
df.info()

## Exploratory Data Analysis (EDA)

### Univarte Analysis

In [ ]:
# Define categorical variables for analysis
cat_var = df[['Company', 'OS', 'Processor', 'TypeName', 'VC']]

# Define numerical variables for analysis
num_var = df[['FS', 'HBD', 'HDD', 'IPS', 'Ram', 'SSD', 'TS']]

In [ ]:
# Assuming num_var is your numerical variable DataFrame
plt.figure(figsize=(12, 15))

# Plotting value count bar plots for each numerical variable
for i, column in enumerate(num_var.columns):
    plt.subplot(3, 3,i+1)
    num_var[column].value_counts().plot(kind='bar')
    plt.title(f'{column}')
    plt.ylabel('Count')
    plt.xticks(rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
# Figure size
plt.figure(figsize=(12, 15))

# Plotting value count bar plots for each categorical variable
for i, column in enumerate(cat_var.columns):
    plt.subplot(3, 2, i+1)
    sns.countplot(data=cat_var, x=column, palette='viridis')  # Use 'viridis' colormap
    plt.title(f'{column}')
    plt.ylabel('Count')
    plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

##  Bivarite Analysis

### Numerical correlation matrix with Target column ('Price')

In [ ]:
# Exclude non-numeric columns
numeric_df = df.select_dtypes(include=['number'])

# Set the figure size
plt.figure(figsize=(12, 10))

# Create a heatmap of the correlation matrix with annotations
sns.heatmap(numeric_df.corr(), annot= True, cmap='viridis', fmt=".2f")

# Set the title of the heatmap
plt.title('Correlation Heatmap')

# Display the heatmap
plt.show()

In [ ]:
df=df.drop(columns=['FS','HBD'])
df

In [ ]:
print(df['HDD'].value_counts())

In [ ]:
# Replace values in the 'HDD' column
df['HDD'] = df['HDD'].replace({1: 1024, 2: 2048, 500:512})

# Verify the changes
print(df['HDD'].value_counts())


In [ ]:
print(df['SSD'].value_counts())

In [ ]:
# Replace values in the 'HDD' column
df['SSD'] = df['SSD'].replace({500: 512, 1: 1024, 2: 2048, 240: 256, 180: 128})

# Verify the changes
print(df['SSD'].value_counts())

### Categorial Correlation with Price

In [ ]:
# Set up subplots
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(18, 25))

# Flatten axes for easy iteration
axes = axes.flatten()

# Plotting boxplots for each categorical variable against Price
for i, column in enumerate(cat_var.columns):
    sns.boxplot(x=cat_var[column], y=df['Price'], ax=axes[i], palette='tab10')  # Set palette to 'viridis'
    axes[i].set_title(f'{column} vs Price')
    axes[i].set_xlabel(column)
    axes[i].set_ylabel('Price')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Models Training 

In [ ]:
# Import necessary libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [ ]:
sns.displot(df['Price'])

In [ ]:
sns.displot(np.log(df['Price']))

In [ ]:
 # Drop the target variable column
X = df.drop('Price', axis=1) 
y = np.log(df['Price'])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('OneHot', OneHotEncoder(sparse= False, drop='first'), [0,1,2,3,4])
],remainder='passthrough')

In [ ]:
# Create pipeline with preprocessing steps and models

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor())
])
pipeline_Gb = Pipeline([
    ('preprocessor', preprocessor),
    ('Gb', GradientBoostingRegressor())
])

pipeline_Xgr = Pipeline([
    ('preprocessor', preprocessor),
    ('Xgr', XGBRegressor())
])

# List of pipelines for easy iteration
pipelines = [pipeline_rf, pipeline_Gb,  pipeline_Xgr]

# Dictionary of pipelines and model types for easy reference
pipe_dict = {0: 'Random Forest', 1: 'Gradient Boosting', 2: 'XGRegressor'}

# Fit the pipelines
for pipe in pipelines:
    pipe.fit(X_train, y_train)

## Models Evaulations

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
# Fit the pipelines and evaluate them
for i, pipe in enumerate(pipelines):
    
    # Generate predictions on the test data
    y_pred = pipe.predict(X_test)
    
    # Calculate the R2 score for the current model
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    # Print the R2 score for the current model
    print(f"{pipe_dict[i]} R2 Score:", r2)
    print(f"{pipe_dict[i]} Mean Squared Error:", mse,'\n')

###  Model's HyperparametersTuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grids for each model
param_grid_rf = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 10, 20, 30],
    'rf__min_samples_split': [2, 5, 10]
}

param_grid_Gb = {
    'Gb__n_estimators': [50, 100, 200],
    'Gb__learning_rate': [0.05, 0.1, 0.2],
    'Gb__max_depth': [3, 5, 7]
}

param_grid_Xgr = {
    'Xgr__n_estimators': [50, 100, 200],
    'Xgr__learning_rate': [0.05, 0.1, 0.2],
    'Xgr__max_depth': [3, 5, 7]
}

# Define GridSearchCV objects for each pipeline
grid_search_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='r2')
grid_search_Gb = GridSearchCV(pipeline_Gb, param_grid_Gb, cv=5, scoring='r2')
grid_search_Xgr = GridSearchCV(pipeline_Xgr, param_grid_Xgr, cv=5, scoring='r2')

# List of grid search objects for easy iteration
grid_searches = [grid_search_rf, grid_search_Gb, grid_search_Xgr]

# Fit the grid search objects
for grid_search in grid_searches:
    grid_search.fit(X_train, y_train)

# Evaluate the best models from grid search
for i, grid_search in enumerate(grid_searches):
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print(f"Best {pipe_dict[i]} R2 Score:", r2)

# Deciding Final Model: Random Forest Regressor

In [ ]:
# Create a pipeline with preprocessing steps and Random Forest Regressor as the final model
pipe = Pipeline([
    ('preprocessor', preprocessor),  # Preprocess the data using the defined preprocessor
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42))  # Random Forest Regressor with 100 estimators and a fixed random state
])

# Fit the final pipeline to the training data
pipe.fit(X_train, y_train)

# The final_pipeline is now trained and ready for making predictions

In [ ]:
# Generate predictions on the test data
y_pred = pipe.predict(X_test)

In [ ]:
# Apply inverse transformation to predictions
y_pred_original_scale = np.exp(y_pred)

print(y_pred_original_scale)

In [ ]:
# Compute performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Print the performance metrics
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("R-squared (R2) Score:", r2)

In [ ]:
# Make predictions on the test set
y_pred = pipe.predict(X_test)

# Plot predicted vs true values
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Random Forest Regressor: True vs Predicted Values')
plt.show()


### Export Model and Data File

In [ ]:
import pickle

# Save the final pipeline to a file using joblib
pickle.dump(pipe, open('pipe.pkl','wb'))

In [ ]:
# Save the cleaned Data file for App
pickle.dump(df,open('df.pkl','wb'))

In [ ]:
df.to_csv('cleaned_Data.csv', index=False)  # Set index=False to exclude the DataFrame index from the CSV